### pyAPES MLM benchmarking at DK-Sor (Soroe Beech Forest)



In [279]:
# setting path
import sys
#sys.path.append('c:\\Repositories\\pyAPES_main')
import os
import importlib
from dotenv import load_dotenv

load_dotenv()
pyAPES_main_folder = os.getenv('pyAPES_main_folder')

sys.path.append(pyAPES_main_folder)
os.chdir(pyAPES_main_folder)  # set working directory so relative file paths resolve correctly
#print(sys.path)

# force iPython re-import modules at each call
%load_ext autoreload
%autoreload 2


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


### Import modules

In [280]:
# force always reload parameters
import pyAPES.parameters.mlm_parameters_Soroe as _soroe_params
importlib.reload(_soroe_params)

from pyAPES.pyAPES_MLM import driver

# import parameter dictionaries for Soroe
from pyAPES.parameters.mlm_parameters_Soroe import gpara, cpara, spara # model configuration, canopy parameters, soil parameters
from pyAPES.utils.iotools import read_forcing, read_results,  read_data

# python packages
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

eps = 1e-16


In [281]:
print(gpara['start_time'], gpara['end_time'])

2019-06-01 2019-10-01


### Read forcing data and compile input for driver


In [282]:
forcing = read_forcing(
    forcing_file=gpara['forc_filename'],
    start_time=gpara['start_time'],
    end_time=gpara['end_time'],
    dt=gpara['dt']
)

params = {
    'general': gpara,   # model configuration
    'canopy': cpara,    # planttype, micromet, canopy, bottomlayer parameters
    'soil': spara,      # soil heat and water flow parameters
    'forcing': forcing  # forging data
}

### Run the model

In [284]:
resultfile, Model = driver(parameters=params,
                           create_ncf=True,
                           result_file= 'Soroe_test.nc'
                          )

INFO pyAPES.pyAPES_MLM driver pyAPES_MLM simulation started. Number of simulations: 1
INFO pyAPES.soil.water __init__ Water balance in soil solved using: RICHARDS EQUATION & no lateral drainage
INFO pyAPES.soil.heat __init__ Soil heat balance solved.
INFO pyAPES.canopy.mlm_canopy __init__ Eflow: True, WMA: False, Ebal: True
INFO pyAPES.microclimate.radiation __init__ Shortwave radiation model: ZHAOQUALLS
INFO pyAPES.microclimate.radiation __init__ Longwave radiation model: ZHAOQUALLS
INFO pyAPES.canopy.forestfloor __init__ Forestfloor has 2 bottomlayer types
INFO pyAPES.utils.iotools save_parameters_yaml Parameters saved to: c:\Repositories\pyAPES_main\input_parameters\Soroe_test.yml
INFO pyAPES.pyAPES_MLM driver Running simulation number (start time 2026-04-20 16:03): 0
INFO pyAPES.pyAPES_MLM run Running simulation 0


0%.. 10%.. 20%.. 30%.. 40%.. 50%.. 60%.. 70%.. 80%.. 90%.. 

INFO pyAPES.pyAPES_MLM run Finished simulation 0, running time 1052.82 seconds
INFO pyAPES.pyAPES_MLM driver Running time 1052.82 seconds


100%


INFO pyAPES.pyAPES_MLM driver Ready! Results are in: c:\\Repositories\\pyAPES_main/results/Soroe_test.nc


### Read model results

In [285]:
# read simulation results to xarray dataset
results = read_results(os.path.join(pyAPES_main_folder, resultfile))

# read observations from DK-Sor
datafile = r'forcing\Soroe\DK-Sor_EC_2015-2020.dat'
data = read_data(os.path.join(pyAPES_main_folder, datafile), start_time=gpara['start_time'], end_time=gpara['end_time'])

#results = read_results(os.path.join(pyAPES_main_folder, r'results\Soroe_test.nc'))
#data = read_data(os.path.join(pyAPES_main_folder, datafile), start_time='2019-03-01', end_time='2019-10-30')

c:\\Repositories\\pyAPES_main/results/Soroe_test.nc


In [ ]:
# results = read_results(os.path.join(pyAPES_main_folder, r'results\Soroe_test.nc'))
# datafile = r'forcing\Soroe\DK-Sor_EC_2015-2020.dat'
# data = read_data(os.path.join(pyAPES_main_folder, datafile), start_time='2019-03-01', end_time='2019-10-30')

In [ ]:
print(results)

# print list of all variables with their dimensions:
vars = list(results.data_vars)
for v in vars:
    print(f"{v}: {results[v].dims}")

### Plot some model results and compare with observations


In [286]:
sim = 0  # in this demo, we have only one simulation (i.e. only one parameter set was used)

# grid variables for plotting
t = results.date  # time
zc = results.canopy_z  # height above ground [m]
zs = results.soil_z  # depth of soil; shown negative [m]

In [ ]:
plt.plot(t, results.canopy_LAI)

### Soil temperature and moisture
- computed using *pyAPES.soil* submodels 'Water' and 'Heat'
- compare with measurements at similar depths (to be done - now compare with Ts and SWC at 5cm depth)

T_depths (m): [0.02, 0.07, 0.17, 0.35]
swc_depths (m): [0.03, 0.09, 0.13, 0.26]


In [ ]:
%matplotlib qt
var = ['soil_temperature', 'soil_volumetric_liquid_water_content']

lyrs = [2, 7, 12, 20] # layers
#depths = np.array2string(np.asarray(zs[lyrs]), precision=1, separator=', ')
depths = ['{:.2f} m'.format(k) for k in zs[lyrs]]

print(depths)
fig, ax = plt.subplots(len(var), 1, figsize=(10,7))

k = 0
ax[0].plot(t, data['TS_F_MDS_1'], 'ko', markersize=3, alpha=0.1)
ax[0].plot(t, data['TS_F_MDS_2'], 'ko', markersize=3, alpha=0.2)
ax[0].plot(t, data['TS_F_MDS_3'], 'ko', markersize=3, alpha=0.4)
ax[0].plot(t, data['TS_F_MDS_4'], 'ko', markersize=3, alpha=0.6)

ax[1].plot(t, 1e-2 * data['SWC_F_MDS_1'], 'ko', markersize=3, alpha=0.1, label='L1')
ax[1].plot(t, 1e-2 * data['SWC_F_MDS_2'], 'ks', markersize=3, alpha=0.2, label='L2')
ax[1].plot(t, 1e-2 * data['SWC_F_MDS_3'], 'kd', markersize=3, alpha=0.4, label='L3')
ax[1].plot(t, 1e-2 * data['SWC_F_MDS_4'], 'kv', markersize=3, alpha=0.6, label='L4')
for v in var:
    ax[k].plot(t, results[v][:,sim,lyrs], label=depths)
    ax[k].set_ylabel(results[v].attrs['units'])
    ax[k].tick_params(axis='x', labelrotation = 20)
    ax[k].legend(fontsize=8)
    k += 1

# # vertical profile at last timestep
# fig, ax = plt.subplots(1, 2) #figsize=(10,5))

# k = 0
# for v in var:
#     ax[k].plot(results[v][-1,sim,:], zs)
#     ax[k].set_xlabel(results[v].attrs['units'])
#     ax[0].set_ylabel('depth (m)')
#     k += 1

In [ ]:

%matplotlib qt
import numpy as np
import matplotlib.pyplot as plt
from pyAPES.soil.water import wrc

# --- pF curves for each soil layer ---
psi_range = -np.logspace(-2, 4.5, 300)  # matric potential [m], negative
psi_cm = 100 * psi_range                 # convert to cm for wrc

layer_labels = ['L1: 0–5 cm', 'L2: 5–15 cm', 'L3: 15 cm – 10 m']
colors_pF = ['tab:brown', 'tab:olive', 'tab:gray']

pF_params = spara['soil_properties']['pF']
n_layers = len(pF_params['ThetaS'])

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# left panel: pF curves
ax = axes[0]
for i in range(n_layers):
    pF_i = {k: pF_params[k][i] for k in pF_params}
    theta = wrc(pF_i, psi=psi_range)
    ax.semilogx(-psi_cm, theta, color=colors_pF[i], lw=2, label=layer_labels[i])

# mark field capacity (pF 2 = 100 cm) and wilting point (pF 4.2 = 15849 cm)
for i in range(n_layers):
    pF_i = {k: pF_params[k][i] for k in pF_params}
    fc = wrc(pF_i, psi=np.array([-1.0]))   # pF 2
    wp = wrc(pF_i, psi=np.array([-150.0])) # pF 4.2
    ax.axhline(float(fc), color=colors_pF[i], lw=0.8, ls='--', alpha=0.6)

ax.axvline(100,   color='k', lw=0.7, ls=':', alpha=0.5); ax.text(110, 0.02, 'FC (pF2)', fontsize=8)
ax.axvline(15849, color='k', lw=0.7, ls=':', alpha=0.5); ax.text(17000, 0.02, 'WP (pF4.2)', fontsize=8)
ax.set_xlabel('|ψ| (cm)')
ax.set_ylabel('θ (m³ m⁻³)')
ax.set_title('van Genuchten retention curves')
ax.legend(fontsize=9)
ax.set_xlim(1e-2, 1e5)

# right panel: observed vs modelled SWC at measurement depths
ax2 = axes[1]
swc_obs_cols = ['SWC_F_MDS_1', 'SWC_F_MDS_2', 'SWC_F_MDS_3', 'SWC_F_MDS_4']
swc_obs_depths = [0.03, 0.09, 0.13, 0.26]   # m (approx measurement depths)
obs_colors = ['#1f77b4','#ff7f0e','#2ca02c','#d62728']

for col, depth, oc in zip(swc_obs_cols, swc_obs_depths, obs_colors):
    ax2.plot(t, 1e-2 * data[col], '.', markersize=2, alpha=0.4, color=oc, label=f'obs {depth} m')

# model: pick layer indices closest to measurement depths
lyrs_swc = [int(np.argmin(np.abs(zs.values - (-d)))) for d in swc_obs_depths]
for lyr, depth, oc in zip(lyrs_swc, swc_obs_depths, obs_colors):
    ax2.plot(t, results['soil_volumetric_liquid_water_content'][:, sim, lyr],
             '-', lw=1.2, color=oc, label=f'mod {depth} m')

ax2.set_ylabel('SWC (m³ m⁻³)')
ax2.set_xlabel('Date')
ax2.set_title('Observed vs modelled SWC')
ax2.tick_params(axis='x', labelrotation=20)
ax2.legend(fontsize=7, ncol=2)

fig.tight_layout()
plt.show()

print('\nCurrent pF parameters:')
print(f"  zh (layer boundaries): {spara['grid']['zh']}")
for i in range(n_layers):
    print(f"  {layer_labels[i]}:  ThetaS={pF_params['ThetaS'][i]:.3f}  ThetaR={pF_params['ThetaR'][i]:.3f}"
          f"  alpha={pF_params['alpha'][i]:.4f}  n={pF_params['n'][i]:.3f}"
          f"  Ksat_v={spara['soil_properties']['saturated_conductivity_vertical'][i]:.2e} m/s")


In [ ]:

%matplotlib qt
import numpy as np
import matplotlib.pyplot as plt
from pyAPES.planttype.rootzone import RootDistribution

# --- root distribution for each planttype ---
dz_soil = np.array(spara['grid']['dz'])
z_soil = -np.cumsum(dz_soil) + dz_soil / 2  # node centres, negative downward [m]

fig, axes = plt.subplots(1, 3, figsize=(14, 5))

# left: root distribution profiles
ax = axes[0]
pt_names = list(cpara['planttypes'].keys())
for pt_name, pt in cpara['planttypes'].items():
    rp = pt['rootp']
    rad = RootDistribution(rp['beta'], dz_soil, rp['root_depth'])
    # rad is per unit depth [m-1], multiply by dz to get fraction per layer
    z_root = z_soil[:len(rad)]
    ax.barh(z_root, rad * dz_soil[:len(rad)], height=dz_soil[:len(rad)] * 0.8,
            label=f"{pt_name}  (depth={rp['root_depth']} m, β={rp['beta']})", alpha=0.6)
ax.set_xlabel('Root fraction per layer (-)')
ax.set_ylabel('Depth (m)')
ax.set_title('Root distribution')
ax.legend(fontsize=8)
ax.axhline(0, color='k', lw=0.8)

# middle: mean seasonal SWC profile – growing season vs. dormant
swc_mod = results['soil_volumetric_liquid_water_content'][:, sim, :].values  # (time, soil)
t_pd = results.date.to_index()
is_growing = (t_pd.month >= 5) & (t_pd.month <= 9)
swc_grow = swc_mod[is_growing].mean(axis=0)
swc_dorm = swc_mod[~is_growing].mean(axis=0)

ax2 = axes[1]
ax2.plot(swc_grow, zs.values, 'g-o', markersize=3, lw=1.5, label='mod growing (May–Sep)')
ax2.plot(swc_dorm, zs.values, 'b-o', markersize=3, lw=1.5, label='mod dormant')
ax2.set_xlabel('Mean SWC (m³ m⁻³)')
ax2.set_ylabel('Depth (m)')
ax2.set_title('Modelled mean SWC profile')
ax2.set_ylim(-1.0, 0.02)
ax2.axhline(0, color='k', lw=0.8)
ax2.legend(fontsize=8)

# right: depth-resolved drying signal  Δθ = θ_dormant − θ_growing  (positive = drying in summer)
delta_swc = swc_dorm - swc_grow
ax3 = axes[2]
ax3.barh(zs.values, delta_swc, height=np.array(dz_soil) * 0.8, color='darkorange', alpha=0.7)
ax3.axvline(0, color='k', lw=0.8)
ax3.set_xlabel('Δθ = dormant − growing (m³ m⁻³)')
ax3.set_ylabel('Depth (m)')
ax3.set_title('Summer drying by depth\n(+ve = drier in growing season)')
ax3.set_ylim(-1.0, 0.02)

# overlay root zone boundary
for ax_ in axes:
    for pt in cpara['planttypes'].values():
        ax_.axhline(-pt['rootp']['root_depth'], color='r', lw=0.8, ls='--', alpha=0.6)
axes[2].text(delta_swc.max() * 0.6, -0.72, 'root depth', color='r', fontsize=7)

fig.suptitle('Root uptake diagnostics', fontsize=12)
fig.tight_layout()
plt.show()

print('\nRoot parameters:')
for pt_name, pt in cpara['planttypes'].items():
    rp = pt['rootp']
    print(f"  {pt_name}: root_depth={rp['root_depth']} m, beta={rp['beta']}, "
          f"root_to_leaf_ratio={rp['root_to_leaf_ratio']}, root_conductance={rp['root_conductance']:.1e}")


In [ ]:

%matplotlib qt
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from pyAPES.soil.water import wrc

pF_params = spara['soil_properties']['pF']
zh = spara['grid']['zh']
dz_soil = np.array(spara['grid']['dz'])
z_nodes = -np.cumsum(dz_soil) + dz_soil / 2

def get_pf_zone(lyr):
    iz = int(np.sum(z_nodes[lyr] < np.array([-abs(z) for z in zh])))
    return min(iz, len(pF_params['ThetaS']) - 1)

def wrc_scalar(iz, psi_m):
    pF_i = {k: pF_params[k][iz] for k in pF_params}
    return float(wrc(pF_i, psi=np.array([psi_m])))

# layers to diagnose: shallow to deep
diag_layers = [
    (lyrs[1], 'SWC_F_MDS_2', 'obs ~9 cm'),
    (lyrs[2], 'SWC_F_MDS_3', 'obs ~13 cm'),
    (lyrs[3], 'SWC_F_MDS_4', 'obs ~26 cm'),
]
colors_lyr = ['tab:blue', 'tab:orange', 'tab:red']

transp = results['soil_transpiration'][:, sim].values
precip = results['forcing_precipitation'][:, sim].values

fig, axes = plt.subplots(5, 1, figsize=(14, 13), sharex=True)
fig.suptitle('Soil water budget diagnostics - multiple depths', fontsize=12)

# panel 1: SWC timeseries for all layers
ax = axes[0]
for (lyr, obs_col, obs_lbl), col in zip(diag_layers, colors_lyr):
    iz = get_pf_zone(lyr)
    z_diag = float(zs[lyr])
    swc_mod = results['soil_volumetric_liquid_water_content'][:, sim, lyr].values
    ThetaR = pF_params['ThetaR'][iz]
    FC = wrc_scalar(iz, -1.0)
    WP = wrc_scalar(iz, -150.0)
    ax.plot(t.values, 1e-2 * data[obs_col].values, '.', markersize=2, alpha=0.3, color=col)
    ax.plot(t.values, swc_mod, '-', lw=1.2, color=col,
            label=f'mod {z_diag:.2f} m  (ThetaR={ThetaR:.2f}, FC={FC:.2f}, WP={WP:.2f})')
    ax.axhline(ThetaR, color=col, lw=0.6, ls='--', alpha=0.5)
ax.set_ylabel('SWC (m3 m-3)')
ax.legend(fontsize=7)

# panel 2: transpiration
axes[1].plot(t.values, transp * 1e3, 'tab:green', lw=1.0, label='root water uptake (mm/s)')
axes[1].set_ylabel('Transp. (mm/s)')
axes[1].legend(fontsize=8)

# panel 3: precipitation
axes[2].bar(t.values, precip * 3600, width=1/48, color='steelblue', alpha=0.7, label='Precip (mm/30min)')
axes[2].set_ylabel('Precip (mm/step)')
axes[2].legend(fontsize=8)

# panel 4: distance to ThetaR floor per layer
for (lyr, obs_col, obs_lbl), col in zip(diag_layers, colors_lyr):
    iz = get_pf_zone(lyr)
    z_diag = float(zs[lyr])
    swc_mod = results['soil_volumetric_liquid_water_content'][:, sim, lyr].values
    ThetaR = pF_params['ThetaR'][iz]
    dist = swc_mod - ThetaR
    axes[3].plot(t.values, dist, color=col, lw=1.0, label=f'z={z_diag:.2f} m')
axes[3].axhline(0, color='k', lw=0.8, ls='--')
axes[3].set_ylabel('theta - ThetaR (m3/m3)\n(0 = hits floor)')
axes[3].legend(fontsize=8)

# panel 5: cumulative precip vs cumulative transpiration
cum_prec  = np.cumsum(precip) * gpara['dt']
cum_trans = np.cumsum(transp) * gpara['dt']
axes[4].plot(t.values, cum_prec  * 1e3, 'steelblue', lw=1.2, label='Cum. precip (mm)')
axes[4].plot(t.values, cum_trans * 1e3, 'tab:green',  lw=1.2, label='Cum. transpiration (mm)')
axes[4].set_ylabel('Cumulative (mm)')
axes[4].set_xlabel('Date')
axes[4].legend(fontsize=8)

for ax_ in axes:
    ax_.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
    ax_.xaxis.set_major_locator(mdates.MonthLocator(interval=1))
    ax_.tick_params(axis='x', labelrotation=20)

fig.tight_layout()
plt.show()

print('\nSummary per layer:')
for lyr, obs_col, obs_lbl in diag_layers:
    iz = get_pf_zone(lyr)
    z_diag = float(zs[lyr])
    swc_mod = results['soil_volumetric_liquid_water_content'][:, sim, lyr].values
    ThetaR = pF_params['ThetaR'][iz]
    FC = wrc_scalar(iz, -1.0)
    WP = wrc_scalar(iz, -150.0)
    min_swc = float(np.nanmin(swc_mod))
    hits_floor = min_swc < ThetaR + 0.005
    print(f'  z={z_diag:.3f} m  zone={iz}  ThetaR={ThetaR:.3f}  FC={FC:.3f}  WP={WP:.3f}  '
          f'PAW={FC-WP:.3f}  min_SWC={min_swc:.3f}  {"*** HITS ThetaR FLOOR ***" if hits_floor else "ok"}')


### Ecosystem-scale fluxes

- ecosystem - atm. fluxes represent the integrated sinks / sources in soil (soil-module), forestfloor (bottomlayer-module) and vegetation (planttype&canopy -modules)
- comparable to ecosystem - atmosphere exchange
- affected by current model forcing and sub-model instance state (e.g. Planttype and Canopy LAI, phenology, soil temperature, moisture etc.)


In [ ]:
data.columns

In [287]:
%matplotlib qt
# var = ['forcing_air_temperature', 'forcing_par', 'canopy_NEE', 'canopy_GPP', 
#        'canopy_Reco', 'canopy_Rnet', 'canopy_SH', 'canopy_LE', 'ffloor_ground_heat']

var = ['canopy_NEE', 'canopy_GPP', 'canopy_Reco']
var2 = ['canopy_Rnet', 'canopy_SH', 'canopy_LE', 'ffloor_ground_heat']


# # temperature & par
# ax[0].plot(t, results[var[0]][:,sim], 'k-', label=var[0])
# ax[0].set_ylabel(results[var[0]].attrs['units'])
# ax[0].tick_params(axis='x', labelrotation = 20)
# axb = ax[0].twinx()

# axb.plot(t, results[var[1]][:,sim], 'r-', label=var[1]) # Par
# axb.set_ylabel(results[var[1]].attrs['units'])

fig, ax = plt.subplots(3, 1, figsize=(8,10), sharex=True)
# CO2 fluxes
ax[0].plot(t, data['NEE_VUT_REF'], 'k.-', alpha=0.3)
ax[1].plot(t, data['GPP_NT_VUT_REF'], 'k.-', alpha=0.3)
ax[1].plot(t, data['GPP_DT_VUT_REF'], 'r.-', alpha=0.3)
ax[2].plot(t, data['RECO_NT_VUT_REF'], 'k.-', alpha=0.3)
ax[2].plot(t, data['RECO_DT_VUT_REF'], 'r.-', alpha=0.3)
n = 0
for v in var:
    ax[n].plot(t, results[v][:,sim], label=v)

    ax[n].set_ylabel('umol m-2 s-1')
    ax[n].tick_params(axis='x', labelrotation = 20)
    ax[n].legend(fontsize=8)
    n +=1

fig, ax = plt.subplots(5, 1, figsize=(8,10), sharex=True)
# energy fluxes
ax[1].plot(t, data['NETRAD'], 'k.-', alpha=0.3)
ax[2].plot(t, data['H_F_MDS'], 'k.-', alpha=0.3)
ax[3].plot(t, data['LE_F_MDS'], 'k.-', alpha=0.3)
ax[4].plot(t, data['G_F_MDS'], 'k.-', alpha=0.3)

n = 1
for v in var2:
    ax[0].plot(t, results[v][:,sim], label=v)
    ax[n].plot(t, results[v][:,sim], label=v)

    ax[n].set_ylabel('W m-2')
    ax[n].tick_params(axis='x', labelrotation = 20)
    ax[n].legend(fontsize=8)
    n += 1


In [ ]:
fig, ax = plt.subplots(4, 1, figsize=(8,10), sharex=True)
# CO2 fluxes
ax[0].plot(t, data['GPP_NT_VUT_REF'], 'k.-', alpha=0.3); ax[0].set_ylabel('GPP (umolm-2s-1)')
ax[0].plot(t, results['canopy_GPP'][:,sim], label='GPP')

ax[0].plot(t, results['ffloor_photosynthesis'], '--')
ax[1].plot(t, data['RECO_NT_VUT_REF'], 'k.-', alpha=0.3); ax[1].set_ylabel('RECO (umolm-2s-1)')
ax[1].plot(t, results['canopy_Reco'][:,sim], label='ER')
ax[1].plot(t, results['ffloor_respiration'][:,sim], '--', label='Rffloor')
ax[1].plot(t, results['ffloor_soil_respiration'][:,sim], 'r--', label='Rsoil')
ax[2].plot(t, data['TS_F_MDS_1'], 'k.-', alpha=0.3); ax[2].set_ylabel('T (degC)')
ax[3].plot(t, 1e-2*data['SWC_F_MDS_2'], 'k.-', alpha=0.3); ax[3].set_ylabel('SWC (m3m-3)')

ax[2].plot(t, results['soil_temperature'][:,sim,lyrs], label=depths)
ax[2].legend(fontsize=8)

var3 = ['soil_volumetric_liquid_water_content']
k = 3
for v in var3:
    ax[k].plot(t, results[v][:,sim,lyrs], label=depths)
    ax[k].set_ylabel(results[v].attrs['units'])
    ax[k].tick_params(axis='x', labelrotation = 20)
    ax[k].legend(fontsize=8)



In [ ]:
print(data.columns)

### Ecosystem radiation balance

- net radiation (Rnet) is the sum of net shortwave (SWnet = incoming - reflected) and net longwave (LWnet = incoming - emitted) radiation at canopy top
- computed via models in pyAPES.microclimate.radiation, called iteratively from pyAPES.canopy.mlm_canopy to account for canopy structure and leaf & forest floor temperature
- ecosystem albedo can be computed from radiation profiles at uppermost grid point

In [ ]:
# net radiation components at canopy top
var = ['canopy_Rnet','canopy_SWnet', 'canopy_LWnet']
profs = ['canopy_par_down', 'canopy_par_up', 'canopy_nir_down','canopy_nir_up','canopy_lw_down','canopy_lw_up']

fig, ax = plt.subplots(3, 1, figsize=(8,12), sharex=True)

for v in var:
    ax[0].plot(t, results[v][:, sim], label=v)
ax[0].set_ylabel('W m-2')
ax[0].tick_params(axis='x', labelrotation = 20)
ax[0].legend(fontsize=8)    

# lets plot also partitioning of canopy_SWnet and canopy_LWnet.
# -1 is the index of uppermost gridpoint

for v in ['canopy_par_down', 'canopy_nir_down','canopy_lw_down']: # downward
    ax[1].plot(t, results[v][:,sim,-1], '-', label=v)
for v in ['canopy_par_up', 'canopy_nir_up','canopy_lw_up']: # upward
    ax[1].plot(t, results[v][:,sim,-1], '--', label=v)
ax[1].set_ylabel('W m-2')
ax[1].tick_params(axis='x', labelrotation = 20)
ax[1].legend(fontsize=8)

# canopy albedo
eps = 1e-16

# fraction of par on total SW
f_par = results['canopy_par_down'][:,sim,-1] / (results['canopy_par_down'][:,sim,-1] + results['canopy_nir_down'][:,sim,-1] + eps)

alb_par = results['canopy_par_up'][:,sim,-1] / (results['canopy_par_down'][:,sim,-1] + eps)
alb_nir = results['canopy_nir_up'][:,sim,-1] / (results['canopy_nir_down'][:,sim,-1] + eps)
alb_sw = f_par * alb_par + (1 - f_par) * alb_nir
alb_sw = np.maximum(0, np.minimum(1.0, alb_sw))

ax[2].plot(t, alb_sw, '-', label='SW albedo')
ax[2].plot(t, alb_par, '-', label='Par albedo')
ax[2].plot(t, alb_nir, '-', label='Nir albedo')

ax[2].tick_params(axis='x', labelrotation = 20)
ax[2].legend(fontsize=8)

In [ ]:
# Explore temperature-related variables in results
temp_vars = [v for v in results.data_vars if 'temp' in v.lower() or 'leaf' in v.lower()]
print("Temperature-related variables:")
for v in temp_vars:
    print(f"  {v}: dims={results[v].dims}, shape={results[v].shape}")

In [ ]:
%matplotlib qt
import numpy as np

# canopy height h = top of planttype[0] LAD profile (highest z where lad > 0)
pt0_lad = list(cpara['planttypes'].values())[0]['lad']
h = float(zc.values[pt0_lad > 0].max())

# find canopy indices closest to z/h = 0.2, 0.6, 1.0
target_ratios = [0.2, 0.6, 1.0]
labels = ['z/h = 0.2 (bottom)', 'z/h = 0.6 (middle)', 'z/h = 1.0 (top)']
colors = ['tab:blue', 'tab:orange', 'tab:green']

idx = [int(np.argmin(np.abs(zc.values - r * h))) for r in target_ratios]
actual_z = [float(zc[i]) for i in idx]
print(f"Canopy height h = {h:.1f} m  (planttype[0]: '{list(cpara['planttypes'].keys())[0]}')")
for r, i, z in zip(target_ratios, idx, actual_z):
    print(f"  z/h = {r} -> index {i}, z = {z:.2f} m")

# compute Tleaf - Tair at each height
Tair = results['canopy_temperature'][:, sim, :]
dT = results['canopy_Tleaf'][:, sim, :] - Tair

fig, axes = plt.subplots(3, 1, figsize=(12, 8), sharex=True)
fig.suptitle('Leaf - air temperature difference (Tleaf - Tair)', fontsize=13)

for ax, i, label, color, z in zip(axes, idx, labels, colors, actual_z):
    ax.plot(t, dT[:, i], color=color, linewidth=0.7, label=f'dT  {label}  (z = {z:.1f} m)')
    ax.axhline(0, color='k', linewidth=0.8, linestyle='--')
    ax.set_ylabel('Tleaf - Tair (deg C)', color=color)
    ax.tick_params(axis='y', labelcolor=color)
    ax.legend(fontsize=9, loc='upper left')
    ax.tick_params(axis='x', labelrotation=20)

    ax2 = ax.twinx()
    ax2.plot(t, Tair[:, i], color='gray', linewidth=0.7, alpha=0.7, label=f'Tair  {label}')
    ax2.set_ylabel('Tair (deg C)', color='gray')
    ax2.tick_params(axis='y', labelcolor='gray')
    ax2.legend(fontsize=9, loc='upper right')

axes[-1].set_xlabel('Date')
fig.tight_layout()
plt.show()


In [ ]:

import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import numpy as np
from pyAPES.soil.heat import sinusoidal_soil_temperature

# --- grid ---
doys = np.arange(1, 366)
dz = np.array([0.05] * 100)
z = dz / 2 - np.cumsum(dz)          # node elevations, negative downward [m]

# --- model parameters ---
T_mean = 8.0
T_amplitude = 10.0
thermal_diffusivity = 5e-7

# --- compute T(z, doy) ---
T_grid = np.array([
    sinusoidal_soil_temperature(z, doy=d, T_mean=T_mean, T_amplitude=T_amplitude, thermal_diffusivity=thermal_diffusivity,)
    for d in doys
]).T  # shape (n_depths, n_doys)

# --- plot ---
fig, ax = plt.subplots(figsize=(10, 5))

levels = np.arange(-6, 18, 1)
cf = ax.contourf(doys, z, T_grid, levels=levels, cmap='RdBu_r', extend='both')
cs = ax.contour(doys, z, T_grid, levels=levels, colors='k', linewidths=0.5, alpha=0.5)
ax.clabel(cs, levels=levels[::2], fmt='%d°C', fontsize=7, inline=True)

cbar = fig.colorbar(cf, ax=ax, label='Soil temperature (°C)')

ax.set_xlabel('Day of year')
ax.set_ylabel('Depth (m)')
ax.set_title(f'Sinusoidal soil heat-wave model  |  $\\bar{{T}}$={T_mean}°C, $A_0$={T_amplitude}°C')
ax.xaxis.set_major_locator(ticker.MultipleLocator(30))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(10))
ax.set_xlim(1, 365)
ax.set_ylim(z[-1], 0)

plt.tight_layout()
plt.show()


In [ ]:

# Sunlit fraction of foliage: time vs. height
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import numpy as np

sunlit = results['canopy_sunlit_fraction'][:, sim, :].values.T  # shape: (canopy, date)
z = zc.values
T = t.values

fig, ax = plt.subplots(figsize=(12, 5))
pcm = ax.pcolormesh(T, z, sunlit, cmap='YlOrRd', vmin=0, vmax=1, shading='auto')
cbar = fig.colorbar(pcm, ax=ax, label='Sunlit fraction (-)')
ax.set_ylabel('Height (m)')
ax.set_xlabel('Date')
ax.set_title('Sunlit fraction of foliage')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=2))
fig.autofmt_xdate()
plt.tight_layout()
plt.show()


In [ ]:

# Sunlit fraction of foliage: time vs. height
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import numpy as np

tmp = results['soil_volumetric_water_content'][:, sim, :].values.T  # shape: (soil, date)
z = zs.values
T = t.values

fig, ax = plt.subplots(figsize=(12, 5))
pcm = ax.pcolormesh(T, z, tmp, cmap='YlOrRd', shading='auto')
cbar = fig.colorbar(pcm, ax=ax, label='m3m-3')
ax.set_ylabel('depth (m)')
ax.set_xlabel('Date')
ax.set_title('vol water content (m3m-3)')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=2))
fig.autofmt_xdate()
plt.tight_layout()
plt.show()

In [ ]:
results.close()